### Manual inspection of victim prediction and matches
- Sampled 15% of victim test dataset stratified by mbfc
- Inspected judge matches

Conclusion: looks good to me

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os

from dotenv import load_dotenv
import dspy

load_dotenv()

openai_key = os.getenv(
        "OPENAI_API_KEY"
    )


/Users/catherine/Library/Caches/pypoetry/virtualenvs/afan-WLhS1US3-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from afan.dataset import load_and_preprocess_dataset

victim_test = load_and_preprocess_dataset(
    dataset_name="victim_test",
    data_dir="../data/test/",
    columns=["ID", "mbfc", "text", "entities"]
)

victim_test.head(2)

,ID,mbfc,text,entities
0,428,left_bias,Scott Pruitt Continues His Favors for Big Poll...,"[Communities near power plants, people from ac..."
1,57,right_bias,Democrats Take a Nonsensical Approach to Clima...,"[the planet, climate response]"


In [4]:
from sklearn.model_selection import train_test_split


# Get stratified sample
_, sampled_victim = train_test_split(
    victim_test,
    test_size=0.15,
    stratify=victim_test['mbfc'],
    random_state=42
)

# Display result
print(f"Original size: {victim_test.shape[0]}")
print(f"Sample size: {sampled_victim.shape[0]}")
print("\nMBFC distribution in sample:")
print(sampled_victim['mbfc'].value_counts())

Original size: 84
Sample size: 13

MBFC distribution in sample:
mbfc
left_bias              4
right_bias             4
left_center_bias       3
questionable_source    2
Name: count, dtype: int64


In [5]:
sampled_victim

,ID,mbfc,text,entities
45,124,left_bias,These 3 supertrees can protect us from climate...,[Life on Earth]
18,279,questionable_source,Greta Thunberg doesn't know how she will retur...,[her mother]
60,478,left_center_bias,Environmental groups to sue Trump administrati...,"[Atlantic Coast, Voters]"
29,75,right_bias,"Gender-based justice, Reparations, Universal i...","[Americans, minority communities]"
26,345,right_bias,EPA Released A Long-Delayed Report Showing Eth...,"[the atmosphere, The environment, water qualit..."
38,847,questionable_source,California districts sue US govt for $1.4bn ov...,"[California drinking water, public health]"
44,239,left_center_bias,Climate change: Democratic lawmaker rips Donal...,[the science behind climate change]
25,452,left_bias,Rhode Island sues major oil companies over cli...,"[Rhode Islanders, Rhode Island]"
35,404,left_bias,EPA To Gut The Only Major Federal Rule To Cut ...,"[environment, economy]"
48,508,left_center_bias,Senior EPA officials collaborated with climate...,"[mainstream climate-change groups, environment..."


# Predict: gpt-4o-mini

In [6]:
prediction_lm = dspy.LM("openai/gpt-4o-mini", api_key=openai_key)

dspy.configure(lm=prediction_lm)

In [7]:
from afan.prompts.signatures import NarrativeArcSignature

predictor = dspy.Predict(NarrativeArcSignature)

In [8]:
from afan.utils import predict

victim_preds = predict(
    df=sampled_victim,
    predictor=predictor,
    entity="victim"
    )

victim_preds.head(2)

,ID,mbfc,text,entities,predicted_entity
45,124,left_bias,These 3 supertrees can protect us from climate...,[Life on Earth],The global environment and all life on Earth t...
18,279,questionable_source,Greta Thunberg doesn't know how she will retur...,[her mother],Future generations affected by climate change


# Judge

In [9]:
judge_lm = dspy.LM("gpt-4.1-mini", api_key=openai_key)

dspy.configure(lm=judge_lm)

In [10]:
from afan.prompts.judges import EntitiesMatchFewShot

judge_few_shot = dspy.Predict(EntitiesMatchFewShot)

In [11]:
from afan.utils import judge

victim_judged = judge(
    df=victim_preds,
    judge_match=judge_few_shot
)

Accuracy: 69.23%


In [12]:
import pandas as pd

pd.set_option('display.max_colwidth', 200)

cols_to_compare = ["ID", "mbfc", "entities", "predicted_entity", "judge_match"]

In [14]:
victim_judged[cols_to_compare][victim_judged.judge_match == True]

,ID,mbfc,entities,predicted_entity,judge_match
45,124,left_bias,[Life on Earth],The global environment and all life on Earth that depend on healthy ecosystems.,True
29,75,right_bias,"[Americans, minority communities]",Low-income communities and marginalized groups affected by climate change and economic disparities.,True
26,345,right_bias,"[the atmosphere, The environment, water quality, wildlife, human health]","The environment, including water bodies, soil quality, and wildlife affected by harmful algae blooms and air pollution.",True
38,847,questionable_source,"[California drinking water, public health]",Residents relying on contaminated groundwater for drinking water,True
44,239,left_center_bias,[the science behind climate change],"The environment and future generations, threatened by climate change and the denial of its science.",True
25,452,left_bias,"[Rhode Islanders, Rhode Island]",Coastal communities and infrastructure in Rhode Island,True
35,404,left_bias,"[environment, economy]",The planet and future generations affected by climate change and pollution.,True
56,84,left_bias,[21 young plaintiffs],"The young plaintiffs, particularly the 21 children involved in the lawsuit, who are affected by climate change.",True
62,46,right_bias,"[downtown Manhattan community, waterfront neighbourhoods, New York City]",Lower Manhattan neighborhoods,True


In [15]:
victim_judged[cols_to_compare][victim_judged.judge_match == False]

,ID,mbfc,entities,predicted_entity,judge_match
18,279,questionable_source,[her mother],Future generations affected by climate change,False
60,478,left_center_bias,"[Atlantic Coast, Voters]",Marine life and the coastal tourism industry in South Carolina.,False
48,508,left_center_bias,"[mainstream climate-change groups, environmental regulation and enforcement]",Scientific integrity and public understanding of climate change,False
52,330,right_bias,[gas prices],The global environment and future generations who will suffer the consequences of climate change if effective measures are not taken.,False
